# 11.5 - Multi-Agent: CrewAI & Agno

**Module 11 - AI Agents** | Netsetos GenAI Engineering

This notebook works through Multi-Agent: CrewAI & Agno hands-on: The multi-agent tax; Roles and specialization; Coordination topologies; CrewAI: role-based crews; Agno: lightweight teams and the four modes; Handoffs and the aggregation problem.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - dependencies

This first cell installs the one third-party package the runnable demos rely on. The pip line is commented out because most cells here are pure Python - you only uncomment it on a fresh Colab or virtual environment.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A one-line environment-prep cell. It installs `python-dotenv` (for reading a `.env` file), left commented so nothing runs unless you are on a clean machine that needs it.

**How the code works - step by step**
- **`# !pip install python-dotenv -q`** - commented install of the dotenv helper; uncomment on Colab or a fresh env.
- The `# noqa` marker tells linters to ignore the magic-style line.

**In one line:** optional install, skipped on a machine that already has the package.

**What you'll see:** (no output - environment prep)

## Setup - API keys

The framework demos (CrewAI, Agno) call a real LLM, so they need a provider key. This cell reads keys from the environment rather than hardcoding them, and seeds empty defaults so imports never crash on a missing variable.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A credentials-loading cell, not a model call. It uses `os.environ.setdefault` so that if a key is already set in your shell it is kept, and if not an empty string is put in its place - any one provider key is enough to start.

**How the code works - step by step**
- **`import os`** - gives access to the process environment.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - keeps an existing OpenAI key, else sets it blank.
- **`...("ANTHROPIC_API_KEY", "")`** and **`...("GOOGLE_API_KEY", "")`** - same for Anthropic and Google; the comments point to each provider's console.

**In one line:** load whatever provider keys exist without ever writing them into the code.

**What you'll see:** (no output - environment prep)

## 1 - The multi-agent tax

Before any team is worth building you have to see what it costs. A team does not cost a little more than one agent - each agent re-reads the task, carries its own role prompt, and passes coordination messages. This cell puts real numbers on that so the rest of the lesson has a price tag.

In [ ]:
# THE MULTI-AGENT TAX: more agents = more tokens. A rough model of why coordination is not free.
def cost(n_agents, task_tokens=200, role_tokens=150, coord_tokens=80):
    per_agent = task_tokens + role_tokens               # each agent re-reads the task + its own role prompt
    coordination = coord_tokens * max(0, n_agents - 1)  # handoff / summary messages between agents
    return n_agents * per_agent + coordination

for n in (1, 3, 5):
    c = cost(n)
    print(f"  {n} agent(s): ~{c:4d} tokens  ({c / cost(1):.1f}x a single agent)")
print("rule: hold the token budget equal and one agent often matches a team -")
print("      pay for extra agents only when the task actually needs them.")

# Output:
#   1 agent(s): ~ 350 tokens  (1.0x a single agent)
#   3 agent(s): ~1210 tokens  (3.5x a single agent)
#   5 agent(s): ~2070 tokens  (5.9x a single agent)
# rule: hold the token budget equal and one agent often matches a team -
#       pay for extra agents only when the task actually needs them.

**Explanation**

A tiny cost model, not a model call. `cost` adds up the tokens a team burns - per-agent work plus coordination overhead - and the loop prints the multiplier as the team grows, making the "3-to-10x" claim concrete.

**How the code works - step by step**
- **`cost(n_agents, ...)`** - computes tokens for a team: `per_agent` (task + role prompt) times `n_agents`, plus `coordination` messages that scale with `n_agents - 1`.
- **`per_agent = task_tokens + role_tokens`** - every agent re-reads the task and its own role prompt on each call.
- **`coordination = coord_tokens * max(0, n_agents - 1)`** - handoff/summary messages between agents.
- **the `for n in (1, 3, 5)` loop** - prints each team's tokens and its ratio to a single agent via `c / cost(1)`.

**In one line:** more agents = per-agent prompts plus coordination, so cost climbs faster than linearly.

**What you'll see:** Three lines: 1 agent ~350 tokens (1.0x), 3 agents ~1210 (3.5x), 5 agents ~2070 (5.9x), then a two-line rule that an equal-budget single agent often matches a team.

## 2 - Roles and specialization

What actually makes an agent a "specialist"? Beyond its tools it is three things: a role, a goal, and a backstory. The backstory is the surprising one - it is a system prompt injected on every call that changes how the agent reasons. This cell proves it by giving two specialists the same question.

In [ ]:
# A SPECIALIST = role + goal + BACKSTORY (+ tools). The backstory is a system prompt that shapes
# HOW the agent reasons - so the same question gets different answers from different specialists.
class Specialist:
    def __init__(self, name, backstory, lens):
        self.name, self.backstory, self.lens = name, backstory, lens
    def answer(self, question):            # SCRIPTED think step - a real agent calls an LLM here
        return f"[{self.name}] {self.lens(question)}"

optimist = Specialist("Optimist", "always sees upside and opportunity",
                      lambda q: "worth it - GenAI skills pay back fast in a growing market")
skeptic  = Specialist("Skeptic",  "questions assumptions and finds the risks",
                      lambda q: "risky - only pays off if you finish it and actually apply it")

q = "Is the 14999 GenAI course worth it?"
for agent in (optimist, skeptic):
    print(" ", agent.answer(q))
print("same question, different answers - the BACKSTORY changed how each agent reasoned")

# Output:
#   [Optimist] worth it - GenAI skills pay back fast in a growing market
#   [Skeptic] risky - only pays off if you finish it and actually apply it
# same question, different answers - the BACKSTORY changed how each agent reasoned

**Explanation**

A demonstration cell that stands in for two real agents with a scripted think step. The `Specialist` class holds a `backstory` and a `lens` function; two instances answer the identical question and disagree, showing the backstory - not the question - drives the answer.

**How the code works - step by step**
- **`Specialist.__init__`** - stores `name`, `backstory` (the stand-in system prompt), and a `lens` function.
- **`answer`** - a scripted think step (`self.lens(question)`); the comment notes a real agent would call an LLM here.
- **`optimist` / `skeptic`** - two specialists whose lenses return opposite takes on the same course.
- **the `for agent in (...)` loop** - asks both the same question `q` and prints each reply.

**In one line:** same question, different backstory, opposite answer.

**What you'll see:** The Optimist prints "worth it - GenAI skills pay back fast" and the Skeptic "risky - only pays off if you finish it", followed by a line noting the backstory changed how each reasoned.

## 3 - Coordination topologies

Given specialists, how do they collaborate? There are four canonical wirings - sequential, route, broadcast, and debate - and picking the right one is the whole art of multi-agent design. This cell runs one cast of agents through all four so you can see the same team behave four different ways.

In [ ]:
# FOUR WAYS A TEAM COORDINATES. Same cast, same goal, four SHAPES. The think step is scripted.
KB = {"genai": "GenAI: 14999 INR, 146 hrs, high demand"}
def researcher(ctx): return "facts: " + KB["genai"]
def analyst(ctx):    return "analysis: strong ROI if applied  <- " + ctx
def writer(ctx):     return "report: enrol; strong ROI if applied"

def sequential(agents):                 # ASSEMBLY LINE: each agent builds on the previous output
    ctx = ""
    for a in agents: ctx = a(ctx)
    return ctx

def route(agents, leader_pick):         # SUPERVISOR/ROUTE: a leader sends the task to ONE best agent
    return agents[leader_pick]("")

def broadcast(agents):                  # BROADCAST: fan out to ALL in parallel, then synthesise
    views = [a(" (parallel)") for a in agents]
    return f"synthesis of {len(views)} views -> enrol (consensus)"

def debate(pro, con):                   # DEBATE: agents critique each other, then a judge decides
    return f"judge: weighed [{pro('')}] vs [{con('')}] -> enrol, but finish it"

team = [researcher, analyst, writer]
print("sequential:", sequential(team))
print("route     :", route(team, 0), "  (only the researcher ran - cheapest)")
print("broadcast :", broadcast([researcher, analyst]))
print("debate    :", debate(lambda c: "pro: high upside", lambda c: "con: only if applied"))

# Output:
# sequential: report: enrol; strong ROI if applied
# route     : facts: GenAI: 14999 INR, 146 hrs, high demand   (only the researcher ran - cheapest)
# broadcast : synthesis of 2 views -> enrol (consensus)
# debate    : judge: weighed [pro: high upside] vs [con: only if applied] -> enrol, but finish it

**Explanation**

A patterns cell with scripted agents (no LLM). Three plain functions play researcher/analyst/writer, and four higher-order functions wire them into the four topologies, so the difference you see is purely the coordination shape.

**How the code works - step by step**
- **`researcher` / `analyst` / `writer`** - scripted agents reading a tiny `KB` dict.
- **`sequential`** - assembly line: threads `ctx` through each agent in turn.
- **`route`** - supervisor picks one agent by index (`leader_pick`) and only that one runs - the cheapest.
- **`broadcast`** - fans the task to all agents in parallel, then reports a synthesis of their views.
- **`debate`** - runs a pro and a con, then a judge weighs both.

**In one line:** same cast, four shapes - route runs one agent, broadcast and debate run many.

**What you'll see:** Four labeled lines: sequential's final report, route's researcher-only facts ("only the researcher ran - cheapest"), broadcast's synthesis of 2 views, and debate's judge weighing pro vs con -> "enrol, but finish it".

## 4 - CrewAI: role-based crews

CrewAI makes the role-based team its core abstraction. You define Agents (role/goal/backstory/tools), write Tasks that can depend on each other, and group them in a Crew with a process. This cell is Step 3's sequential topology rebuilt in a real framework.

In [ ]:
# CREWAI - role-based crews. Each agent has a role/goal/BACKSTORY; a Crew runs the tasks in order.
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

@tool
def search_courses(topic: str) -> str:
    "Search the Netsetos course catalog by topic."
    return {"genai": "GenAI: 14999 INR, 146 hrs, high demand"}.get(topic.lower(), "not found")

researcher = Agent(role="Course Researcher", goal="Find the course + demand facts",
                   backstory="An edtech researcher who digs up the numbers.",   # backstory shapes reasoning
                   tools=[search_courses], llm="anthropic/claude-sonnet-4-6")
writer = Agent(role="Advisor", goal="Write a clear recommendation",
               backstory="A student-friendly writer who turns facts into advice.",
               llm="anthropic/claude-sonnet-4-6")

research = Task(description="Research the GenAI course and its demand.",
                expected_output="Key facts + demand.", agent=researcher)
advise = Task(description="Write a short recommendation from the research.",
              expected_output="A 3-line advisory.", agent=writer, context=[research])

crew = Crew(agents=[researcher, writer], tasks=[research, advise], process=Process.sequential)
result = crew.kickoff()          # sequential: the researcher's output feeds the writer's task
print(result)

# Output: (illustrative minimal example - needs `pip install crewai crewai-tools` + a key.) The Crew runs
#   the tasks in order; context=[research] hands the researcher's output to the writer. Swap to
#   process=Process.hierarchical (with a manager_llm) to let a manager delegate instead of a fixed line.

**Explanation**

An illustrative CrewAI example - it needs `pip install crewai crewai-tools` plus an API key to actually run. It defines a two-agent crew (researcher + writer) where the writer's task depends on the researcher's output, and `kickoff()` runs the sequential pipeline.

**How the code works - step by step**
- **`@tool search_courses`** - a CrewAI tool the researcher can call to look up a course.
- **`researcher` / `writer` Agents** - each with `role`, `goal`, and a `backstory` that shapes reasoning; the researcher gets the tool.
- **`research` / `advise` Tasks** - `advise` sets `context=[research]`, handing the researcher's output to the writer.
- **`Crew(process=Process.sequential)`** - runs the tasks in order; the comment notes swapping to `Process.hierarchical` (with a `manager_llm`) lets a manager delegate.
- **`crew.kickoff()`** - starts the run.

**In one line:** Agents + Tasks + a sequential Crew = Step 3's assembly line in CrewAI.

**What you'll see:** > **Needs an API key** (set ANTHROPIC_API_KEY) and `pip install crewai crewai-tools`. Illustrative - without a key it does not run; with one it prints the writer's short recommendation built from the researcher's facts.

## 5 - Agno: lightweight teams and the four modes

Agno takes a lighter, mode-first approach: build Agents, group them in a Team, and the `mode` argument *is* the coordination topology. Switching topology is a one-word change. This cell builds a two-member team in `route` mode.

In [ ]:
# AGNO - lightweight teams. A Team has `members` and a `mode`; here `route` sends each query to ONE agent.
from agno.agent import Agent
from agno.team import Team
from agno.models.anthropic import Claude

def search_courses(topic: str) -> str:
    "Search the Netsetos course catalog by topic."
    return {"genai": "GenAI: 14999 INR, 146 hrs"}.get(topic.lower(), "not found")

researcher = Agent(name="Researcher", role="Find course facts",
                   model=Claude(id="claude-sonnet-4-6"), tools=[search_courses])   # tools passed directly
advisor = Agent(name="Advisor", role="Recommend a course to the student",
                model=Claude(id="claude-sonnet-4-6"))

team = Team(name="Advisory", members=[researcher, advisor],   # members=, not agents=
            mode="route",                                     # coordinate | route | broadcast | tasks
            model=Claude(id="claude-sonnet-4-6"))             # the team leader
team.print_response("What does the GenAI course cost?")

# Output: (illustrative minimal example - needs `pip install agno anthropic` + a key.) mode="route" makes the
#   leader dispatch to the single best member (the Researcher). Other modes: coordinate (delegate + synthesise),
#   broadcast (all members in parallel), tasks (decompose into units). Agno is model-agnostic - swap Claude for any.

**Explanation**

An illustrative Agno example - it needs `pip install agno anthropic` plus a key to run. It shows Agno's shape: agents with tools passed directly, a `Team(members=..., mode=...)`, and a leader model, with `route` dispatching each query to the single best member.

**How the code works - step by step**
- **`search_courses`** - a plain function passed straight to the agent as a tool (no decorator needed).
- **`researcher` / `advisor` Agents** - each with a `name`, `role`, and a `Claude` model; the researcher gets the tool.
- **`Team(members=[...], mode="route", model=...)`** - note `members=` (not `agents=`); `mode` is the topology and `model` is the team leader.
- **`team.print_response(...)`** - runs the team on a question; `route` sends it to the one best member.
- The comment lists the other modes: `coordinate`, `broadcast`, `tasks`.

**In one line:** a Team of `members` plus a one-word `mode` = Step 3's topologies as a single switch.

**What you'll see:** > **Needs an API key** (set ANTHROPIC_API_KEY) and `pip install agno anthropic`. Illustrative - with a key, `route` mode has the leader dispatch to the Researcher and prints its response; without one it does not run.

## 6 - Handoffs and the aggregation problem

Agents collaborate by handing off work, and how they hand off matters - full-history vs clean-slate. But the trap that sinks naive teams is the aggregation problem: parallel agents on coupled work each see half the picture and make clashing choices someone must reconcile. This cell stages that conflict.

In [ ]:
# HANDOFFS + THE AGGREGATION PROBLEM. Two coders work IN PARALLEL on partial context and pick
# clashing choices; a later step has to reconcile them (Cognition: do not parallelise coupled work).
def coder_a(spec): return {"button": "Buy Now",  "color": "green"}   # saw only half the spec
def coder_b(spec): return {"button": "Purchase", "color": "blue"}    # saw the other half

a, b = coder_a("checkout"), coder_b("checkout")
conflict = {k: (a[k], b[k]) for k in a if a[k] != b[k]}
print("agent A:", a)
print("agent B:", b)
print("CONFLICT (parallel agents on partial context):", conflict)

# Fix: a SUPERVISOR with the full context decides once, consistently.
resolved = {"button": "Buy Now", "color": "green"}
print("resolved by a supervisor / shared context:", resolved)
print("lesson: parallel agents on COUPLED work make conflicting choices; keep coupled work sequential")

# Output:
# agent A: {'button': 'Buy Now', 'color': 'green'}
# agent B: {'button': 'Purchase', 'color': 'blue'}
# CONFLICT (parallel agents on partial context): {'button': ('Buy Now', 'Purchase'), 'color': ('green', 'blue')}
# resolved by a supervisor / shared context: {'button': 'Buy Now', 'color': 'green'}
# lesson: parallel agents on COUPLED work make conflicting choices; keep coupled work sequential

**Explanation**

A demonstration cell (scripted, no LLM) of why coupled work should not be parallelized. Two coders each return their own choices for the same button, the code diffs them to surface the clash, then shows a supervisor with full context resolving it.

**How the code works - step by step**
- **`coder_a` / `coder_b`** - two agents that each saw only half the spec and picked a different button label and color.
- **`conflict = {k: (a[k], b[k]) for k in a if a[k] != b[k]}`** - builds a dict of exactly the keys where their parallel choices disagree.
- **`resolved`** - a supervisor with the full context decides once, consistently.
- **the print lines** - show each agent's output, the conflict, the resolution, and the lesson.

**In one line:** parallel agents on partial context clash; a supervisor with full context reconciles.

**What you'll see:** Agent A's and B's dicts, then a CONFLICT dict showing button ('Buy Now','Purchase') and color ('green','blue'), the supervisor-resolved answer, and a line that coupled work should stay sequential.

## 7 - When multi-agent wins (and when it hurts)

The closing judgment, turned into a decision function. Multi-agent wins on parallel, isolation, and critic tasks; it hurts on coupled, sequential work where every step constrains the next. This cell encodes those rules and runs four realistic tasks through them.

In [ ]:
# WHEN TO GO MULTI-AGENT - default to a single agent; escalate only when the task earns it.
def choose(sequential_pipeline, parallelizable, needs_isolation, needs_critic):
    if sequential_pipeline: return "single agent", "each step constrains the next; parallel agents would conflict (Cognition)"
    if parallelizable:      return "multi-agent",  "independent subtasks run in parallel (breadth-first research)"
    if needs_isolation:     return "multi-agent",  "isolate contexts so one subtask does not pollute another"
    if needs_critic:        return "multi-agent",  "a separate reviewer catches the worker's mistakes"
    return "single agent", "one well-designed agent is cheaper and usually enough"

cases = [
    ("write a feature end to end", dict(sequential_pipeline=1, parallelizable=0, needs_isolation=0, needs_critic=0)),
    ("research 8 sources at once", dict(sequential_pipeline=0, parallelizable=1, needs_isolation=0, needs_critic=0)),
    ("draft then review code",     dict(sequential_pipeline=0, parallelizable=0, needs_isolation=0, needs_critic=1)),
    ("answer one simple question", dict(sequential_pipeline=0, parallelizable=0, needs_isolation=0, needs_critic=0)),
]
for name, flags in cases:
    pick, why = choose(**flags)
    print(f"  {name:28s} -> {pick:12s} ({why})")

# Output:
#   write a feature end to end   -> single agent (each step constrains the next; parallel agents would conflict (Cognition))
#   research 8 sources at once   -> multi-agent  (independent subtasks run in parallel (breadth-first research))
#   draft then review code       -> multi-agent  (a separate reviewer catches the worker's mistakes)
#   answer one simple question   -> single agent (one well-designed agent is cheaper and usually enough)

**Explanation**

A decision-rule cell (no LLM) that captures the whole lesson's economics. `choose` checks task properties in priority order and returns single-vs-multi with a one-line reason grounded in the Cognition and Anthropic findings; the loop applies it to four cases.

**How the code works - step by step**
- **`choose(...)`** - checks flags in order: a `sequential_pipeline` -> single agent (parallel would conflict); else `parallelizable` / `needs_isolation` / `needs_critic` -> multi-agent; else the single-agent default.
- **`cases`** - four tasks with their property flags set: write a feature, research 8 sources, draft-then-review, a simple question.
- **the `for name, flags in cases` loop** - unpacks each case with `choose(**flags)` and prints the pick plus the reason.

**In one line:** coupled/sequential -> single; parallel/isolation/critic -> multi; default -> single.

**What you'll see:** Four aligned rows: "write a feature end to end" and "answer one simple question" -> single agent; "research 8 sources at once" and "draft then review code" -> multi-agent, each with its reason.

Across seven runnable and illustrative cells you moved from the cost of a team (the 3-to-10x token tax) through what makes a specialist (role + goal + backstory), the four coordination topologies, and the same patterns packaged in CrewAI and Agno - then the two hard-won judgments: the aggregation problem that punishes parallel agents on coupled work, and the rule for when a team actually earns its tax. The throughline is economic: default to one great agent, and add agents only for genuinely parallel, isolation, or critic work. Next up is computer-use agents (11.6), human-in-the-loop gates for teams (11.10), and catching multi-agent failures in agent evaluation (11.11).